# Cameo Requirements Extraction + Artifact Search — Chained Jobs

A recipe for uploading a Cameo `.mdzip` file, running an extraction job, and then **searching for a specific artifact by name or keyword** without knowing the model name or artifact IDs up front.

It uses [`istari_fluent`](../fluent), an opinionated, chainable wrapper over the official [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup).

### What we cover

- Connecting to the platform with a Personal Access Token.
- Registering a Cameo `.mdzip` file as a Model.
- Running an extraction job to produce artifacts.
- **Searching for a specific artifact using two approaches:**
  - Structural filter via `platform.resources()` — for when you know the filename or display name.
  - Full-text search via `platform.client.search_resources()` — for keyword search across all metadata fields.
- Reading the found artifact's content.

### Prerequisites

- An **Istari Digital Platform account** and a **Personal Access Token**.
- An agent with the **Cameo** integration and access to `@istari:extract`.
- A Cameo `.mdzip` file to extract from.

### 1 &middot; Credentials

Create a `.env` file **next to this notebook** with:

```
ISTARI_REGISTRY_URL=https://...paste your platform's registry URL here...
ISTARI_PERSONAL_ACCESS_TOKEN=...paste your token here...
```

### 2 &middot; Install dependencies

```bash
cd fluent
uv sync --extra experiment
```

### 3 &middot; Register the venv as a Jupyter kernel

```bash
uv run python -m ipykernel install --user --name istari-fluent --display-name "Python (istari_fluent)"
```

> **A note on `istari_fluent`** — this is a productivity layer maintained alongside the official SDK. It is not the officially supported client. For production integrations, keep the core [`istari-digital-client`](https://docs.istaridigital.com/developers/SDK/01-setup) as your source of truth.

## 1 &middot; Connect and verify

`IstariPlatform.from_env()` reads `ISTARI_REGISTRY_URL` and `ISTARI_PERSONAL_ACCESS_TOKEN` and returns an `IstariPlatform` object.

In [23]:
from pathlib import Path

from istari_fluent import IstariPlatform, JobDefinition, ResourceView

platform = IstariPlatform.from_env()

# Corporate network with an internal CA bundle? Point from_env() at your .pem:
# platform = IstariPlatform.from_env(ca_bundle="/path/to/ca.pem")

report = platform.client.readiness_check()
assert report.healthy, f"Platform reports unhealthy: {report}"

print(platform)

2026-05-20 15:16:41 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
2026-05-20 15:16:41 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.
2026-05-20 15:16:41 -  istari_digital_client.compatibility:process_response_headers:81 - WARNING - SDK is incompatible with Istari Registry v10.15.2 (affected APIs: v2_main:Systems). Please update your SDK to match the server version.


IstariPlatform connected to https://fileservice-v2.demo.istari.app


## 2 &middot; Configure job parameters

Set the path to your Cameo `.mdzip` file, the target Cameo version and OS, and the artifact filename you want to search for after extraction.

To see available functions and supported tool versions on your platform:
```python
functions = platform.client.list_functions(tool="dassault_cameo")
for f in functions.items:
    print(f.name, f.tool_versions, f.operating_systems)
```

In [24]:
# --- Configure these for your environment ---
MDZIP_PATH        = Path.cwd() / "NCXTable-example.mdzip"  # path to your Cameo file
TOOL_VERSION      = "2024x-refresh2"                        # adjust to your Cameo version
OPERATING_SYSTEM  = "Windows 11"                            # adjust to your agent's OS
DISPLAY_NAME      = "NCXTable-example-search.mdzip"
EXTERNAL_ID       = "cameo-extract-and-search-demo"

# Artifact to search for — used in Section 6
SEARCH_FILENAME   = "requirements.json"   # exact filename produced by @istari:extract
SEARCH_KEYWORD    = "requirements"        # keyword for full-text search
# --------------------------------------------

assert MDZIP_PATH.exists(), f"Cameo file not found: {MDZIP_PATH}"
print(f"Using model file: {MDZIP_PATH}")
print(f"Tool version:     {TOOL_VERSION}")
print(f"Operating system: {OPERATING_SYSTEM}")

Using model file: /Users/craighahn/Documents/GitHub/cookbook/istari-digital-client-cookbook/samples/NCXTable-example.mdzip
Tool version:     2024x-refresh2
Operating system: Windows 11


## 3 &middot; Register the Cameo file as a Model Resource

Registering a local file creates a **Resource** of type **model** with a stable identity and revision history.

In [25]:
model = platform.upload_model(
    MDZIP_PATH,
    external_id=EXTERNAL_ID,
    display_name=DISPLAY_NAME,
)
print(f"Uploaded new model {model.id}")
print(model)

Uploaded new model 58bc7417-c594-422b-9ec1-039bc3c15ea0
Model('NCXTable-example-search.mdzip', filename='NCXTable-example.mdzip', id=58bc7417-c594-422b-9ec1-039bc3c15ea0, file=723a08d0-9f16-4141-bb9d-d590f0ec04d0, rev=79b2850f-e348-4d2c-a8b6-33e637865a81)


## 4 &middot; Run the extraction job

A **Job** instructs the platform to run a function from a tool against a Model revision. Here we run `@istari:extract` with the `dassault_cameo` tool.

1. `model.submit_job(definition)` returns a `JobView` immediately — the job is queued.
2. `job.wait(on_poll=...)` blocks until the job reaches a terminal state.
3. `.on_success()` raises `RuntimeError` if the job ended in `FAILED`.

> **Shortcut:** `model.run_job(definition)` submits + waits + checks success in one call.

In [28]:
extract = JobDefinition(
    function="@istari:extract",
    tool_name="dassault_cameo",
    # tool_version=TOOL_VERSION,
    # operating_system=OPERATING_SYSTEM,
)

job = model.submit_job(extract)
print(f"Submitted job {job.id}; polling...")

job.wait(
    timeout=600,
    on_poll=lambda j: print(f"  [{j.status}] id={j.id}"),
).on_success()

print(f"\nJob finished: {job.status}")

Submitted job 46f721aa-5220-4444-ad13-91fe26426b54; polling...
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Pending] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Claimed] id=46f721aa-5220-4444-ad13-91fe26426b54
  [Failed] id=46f721aa-5220-4444-ad13-91fe26426b54


RuntimeError: Job 46f721aa-5220-4444-ad13-91fe26426b54 did not complete (status=Failed)

## 5 &middot; Inspect the job products

`job.get_products()` returns `ResourceView` objects pinned to the exact revisions the job wrote. This is the direct path when you already have a `JobView` in hand.

In [17]:
products = job.get_products()
print(f"Job wrote {len(products)} product(s):\n")
for p in products:
    print(f"  - {p.type:10s}  name={p.name!r:40s}  file={p.file_id}  rev={p.revision_id}")

Job wrote 8 product(s):

  - Artifact    name='istari-module-stdout.txt'                file=c548363c-e3c7-4cc3-a0fa-ca47ea4ec96a  rev=456635c9-89f4-4401-a3f9-bf7d16c0b541
  - Artifact    name='istari-agent-module-exit-status.txt'     file=dcbc3f27-3220-4779-99af-0f9aebf4cb12  rev=2e3b2f25-3e82-4de7-b65c-b5eb882dc90c
  - Artifact    name='istari-module-stderr.txt'                file=c7ff455c-6391-4f07-a7cb-0eb83268818f  rev=9fcee75b-491e-4bbe-8cf1-bfa7668a094c
  - Artifact    name='requirements.json'                       file=dbb23cf3-5888-48d6-92e7-469431b4d9d4  rev=c886bca9-e629-4301-8d2b-ceef7592e68d
  - Artifact    name='relationships.json'                      file=ad045c56-5523-49a0-be8a-7163cca0e7ba  rev=a53edc2c-b040-4cfa-a9f4-1caa401fbf9d
  - Artifact    name='blocks.json'                             file=f4452ec9-0d20-4823-a84c-357c504cab9f  rev=1ec70f92-8d67-4833-ba08-1433b283f2f0
  - Artifact    name='other_elements.json'                     file=733ebb98-bcfc-4cf4-9ffc-b

## 6 &middot; Search for a specific artifact (no model name or artifact ID required)

The sections above walked the job's own product list. In practice you often need to find an artifact **after the fact** — from a different script, a different session, or without any reference to the job that produced it.

Two approaches are shown below:

| Approach | When to use |
|---|---|
| `platform.resources()` filter | You know the exact filename, display name, or external identifier |
| `platform.client.search_resources()` | You have a keyword and want to search across all metadata fields |

Both return results platform-wide — **no model ID or artifact ID needed**.

### Approach A — structural filter via `platform.resources()`

`platform.resources()` returns a lazy `ResourceQuery`. Chain `.type("artifact")` to restrict to artifacts, then `.filter(file_name=...)` to match on filename. A single `file_name` value performs a SQL `LIKE` comparison, so partial names work.

Other useful filters: `display_name`, `description`, `version_name`, `external_identifier`, `mime_type`, `archive_status`.

In [29]:
# Find all artifacts whose filename matches SEARCH_FILENAME.
# Nothing hits the network until .all() / .first() / iteration.
# file_name must be passed as a list (single value performs a LIKE comparison).
matches = (
    platform.resources()
    .type("artifact")
    .filter(file_name=[SEARCH_FILENAME])
    .all()
)

print(f"Found {len(matches)} artifact(s) named '{SEARCH_FILENAME}':\n")
for item in matches:
    print(f"  id={item.id}  name={item.name!r}  updated={item.updated}")

Found 22 artifact(s) named 'requirements.json':

  id=9175bb49-0eb4-4e78-b894-8481c1e11e8e  name='requirements.json'  updated=2026-05-20 17:55:39.463779+00:00
  id=3915dc47-5694-4c9f-9ed5-087190fbd74b  name='requirements.json'  updated=2026-05-20 17:21:14.019939+00:00
  id=c46fc67f-3949-4876-84a9-64fd31a32602  name='requirements.json'  updated=2026-05-20 17:17:35.249842+00:00
  id=dfd5b30b-80a3-4cbb-8527-687264c17524  name='requirements.json'  updated=2026-05-20 17:13:16.901222+00:00
  id=8da59916-9bed-45e2-97a9-f852ed5c4b85  name='requirements.json'  updated=2026-05-20 17:07:09.501132+00:00
  id=f1e63007-0f3c-494b-92c2-5ab3cbb84c8a  name='requirements.json'  updated=2026-05-20 16:58:55.889814+00:00
  id=2c962503-ed73-4bb7-a014-a0712d6dedc8  name='requirements.json'  updated=2026-05-19 21:57:24.274954+00:00
  id=2303e464-3e5e-4d84-ac52-7d02b7dab728  name='requirements.json'  updated=2026-05-18 22:51:50.136486+00:00
  id=fdc4ebf6-07b2-4ad0-87d0-2e4f253f004f  name='requirements.json'  up

### Approach B — full-text search via `platform.client.search_resources()`

`search_resources` performs a full-text search across `name`, `description`, `display_name`, `version_name`, and `external_identifier` simultaneously. Use it when you have a keyword but don't know the exact filename.

Constraints: `search_term` ≥ 1 character; `page` ≥ 1; `size` between 1 and 100.

In [30]:
from istari_digital_client.v2.models import FullTextSearch, ResourceType

# search_resources does full-text search but has no server-side type filter —
# it returns all resource types (models, artifacts, jobs, comments) mixed
# together across pages.  We collect every page and then keep only artifacts.
artifact_hits = []
page = 1
while True:
    batch = platform.client.search_resources(
        full_text_search=FullTextSearch(
            search_term=SEARCH_KEYWORD,
            page=page,
            size=100,   # max allowed
        )
    )
    artifact_hits += [
        r for r in batch.items
        if r.type_name == ResourceType.ARTIFACT
    ]
    if len(artifact_hits) > 0 or page * 100 >= batch.total:
        break
    page += 1

print(f"Full-text search for '{SEARCH_KEYWORD}' returned {batch.total} total hit(s) across all types.")
print(f"{len(artifact_hits)} of those are artifacts:\n")
for item in artifact_hits:
    print(f"  id={item.id}  name={item.name!r}  updated={item.updated}")


Full-text search for 'requirements' returned 82 total hit(s) across all types.
49 of those are artifacts:

  id=74e04f84-8b09-4e73-95ba-17af224e690a  name='requirements.json'  updated=2026-05-11 22:30:28.182376+00:00
  id=bb9e0c20-38d9-4b7a-b0d5-d398bff2fc3e  name='Requirements.csv'  updated=2026-05-20 12:41:53.154301+00:00
  id=18d5cfd4-f4d0-4864-afb4-aa28e5184969  name='Requirements.csv'  updated=2026-05-05 21:06:02.813231+00:00
  id=02c19acf-949e-49db-978a-0c33a06354b3  name='Requirements.csv'  updated=2026-05-19 20:51:31.098679+00:00
  id=9dd8ef28-270c-4bc9-a161-1574827eae64  name='requirements.json'  updated=2026-05-07 02:17:34.998310+00:00
  id=9e6b2607-cec9-4fe6-b4f0-092116232a13  name='Requirements.csv'  updated=2026-05-20 16:03:12.960710+00:00
  id=5b500846-7ab6-45e6-a196-4d0a77a79d75  name='Requirements.csv'  updated=2026-05-20 12:42:24.321780+00:00
  id=5ef6d852-e147-4852-b1ed-7297f88e635b  name='Requirements.csv'  updated=2026-05-12 22:24:28.300871+00:00
  id=7aeb6054-fc7b-

### Pick the target artifact and read its content

Both approaches above return `ResourceSearchItem` objects — lightweight index records with metadata but no file content. To read the actual bytes, fetch the full artifact and wrap it in a `ResourceView` (which provides `read_json()`, `read_text()`, and `download()`).

Here we prefer the most-recently-updated match from Approach A, falling back to Approach B if Approach A found nothing.

In [20]:
# Pick the most-recently-updated artifact from whichever search found results.
candidates = matches or artifact_hits
assert candidates, (
    f"No artifacts found for filename='{SEARCH_FILENAME}' or keyword='{SEARCH_KEYWORD}'. "
    "Check that the extraction job completed and the search terms match."
)

# Sort by updated timestamp descending and take the newest.
target_item = sorted(candidates, key=lambda r: r.updated, reverse=True)[0]
print(f"Selected artifact: id={target_item.id}  name={target_item.name!r}")

# Fetch the full artifact object and wrap it in a ResourceView so we can read its content.
artifact_obj = platform.client.get_artifact(artifact_id=target_item.id)
artifact_view = ResourceView(_resource=artifact_obj, _client=platform.client)

print(f"\nResourceView: {artifact_view}")
print(f"Revision:     {artifact_view.revision_id}")

Selected artifact: id=dfd5b30b-80a3-4cbb-8527-687264c17524  name='requirements.json'

ResourceView: Artifact('requirements.json', id=dfd5b30b-80a3-4cbb-8527-687264c17524)
Revision:     c886bca9-e629-4301-8d2b-ceef7592e68d


### Read the artifact content

`read_json()` downloads and parses the file in one call. Use `read_text()` for plain text or `download(dest)` to write to disk.

In [21]:
requirements_data = artifact_view.read_json()

print(f"Parsed {len(requirements_data)} requirement(s):\n")
for req in requirements_data:
    print(f"  id={req['id']}")
    print(f"  name={req['name']}")
    print(f"  tags={req.get('tags', {})}")
    print()

Parsed 8 requirement(s):

  id=_2021x_2_38b206bc_1744829738919_924302_3837
  name=Requirement REQ-001 Plate Thickness
  tags={'Text': 'The base plate (PLATE_1) thickness shall be between 26 mm and 30 mm to provide adequate bending stiffness while remaining within the prescribed mass budget for the test assembly.', 'Id': '1545599', 'TBD/TBR': False}

  id=_2021x_2_38b206bc_1744829804820_164896_3864
  name=Requirement REQ-002 Plate Length
  tags={'Text': 'The base plate (PLATE_1) length along the X axis shall be between 120 mm and 160 mm to fit within the allocated structural bay.', 'Id': '1545600', 'TBD/TBR': False}

  id=_2022x_2_24f90527_1750033427003_556537_3365
  name=Requirement REQ-003 Plate Width
  tags={'Text': 'The base plate (PLATE_1) width along the Y axis shall be between 120 mm and 160 mm to match the length constraint and maintain a square footprint for symmetric loading.', 'Id': '1545601', 'TBD/TBR': False}

  id=_2024x_2_1fd504ba_1777994538882_283919_2916
  name=Requirem

## 7 &middot; Narrowing a search with multiple filters

When a keyword or filename matches many artifacts (e.g. across many model revisions), combine filters to zero in on the one you want.

All filter parameters that map to list-typed API fields (`file_name`, `display_name`, `external_identifier`, `version_name`, `mime_type`, etc.) **must be passed as Python lists**, even when filtering on a single value.

```python
# Narrow by filename AND external identifier
platform.resources()
    .type("artifact")
    .filter(
        file_name=["requirements.json"],
        external_identifier=["requirements.json],
    )
    .first()

# Only artifacts you created, sorted newest first
platform.resources()
    .type("artifact")
    .filter(file_name=["requirements.json"])
    .sort("-created")
    .first()

# Full-text search restricted to a display name prefix
platform.resources()
    .type("artifact")
    .filter(display_name=["NCXTable"])
    .all()
```

Queries are lazy and immutable — each `.filter()` / `.sort()` / `.type()` call returns a new query object, so a base query can be safely forked and reused without side effects.

In [31]:
# Example: find the requirements.json that belongs to the model we just uploaded,
# using external_identifier as the tie-breaker.
# All list-type filter params (file_name, display_name, external_identifier, ...) must be lists.
precise_match = (
    platform.resources()
    .type("artifact")
    .filter(
        file_name=[SEARCH_FILENAME],
        external_identifier=["requirements.json"],
    )
    .sort("-created")
    .first()
)

if precise_match:
    print(f"Precise match: id={precise_match.id}  name={precise_match.name!r}")
else:
    print(
        "No precise match found. The platform may not propagate external_identifier "
        "to job-produced artifacts automatically — use the filename-only filter instead."
    )

Precise match: id=9175bb49-0eb4-4e78-b894-8481c1e11e8e  name='requirements.json'


## Verify in the UI

Sign in to the same platform you used for your token and cross-check:

1. **Files / Models** — You should see the model with the display name set in Step 3.
2. **Jobs / Activity** — One `@istari:extract` job for `dassault_cameo`.
3. **Resources / Artifacts** — Search for `requirements.json` and confirm it appears with the correct revision.

## Optional &middot; Archive the model

Archiving hides the Model from default listings in the UI and API. The data and lineage stay intact — this is a soft delete you can reverse later.

In [ ]:
model.archive()